# OLAP · 07 · Findings, recommendations and charts
**Owner: Ana** · Reviewer: M · Runs after: OLAP notebooks 01–06

This notebook does **no new analysis**. It reads the answers saved by notebooks 01–06 in the `olap` schema, draws one chart per finding, and states what Kestrel General Insurance should do about each one.

**Scope of the evidence:** 161 in-scope ICA catastrophe events from January 2011 to March 2026, $56.3bn of normalised insured loss (2022 dollars), joined to ABS population and median income by state.

| # | Finding | Business question | Evidence |
|---|---|---|---|
| F1 | Flood and hail cause almost two-thirds of loss | BQ1 | OLAP 01 · 1.1, 1.2 |
| F2 | A dozen events make half the loss; a bad year is more than twice an average one | BQ1 | OLAP 01 · 1.3, 1.4 |
| F3 | Queensland and NSW carry 77% of loss; Queensland is the outlier per resident | BQ3 | OLAP 03 · 3.1–3.4 |
| F4 | The apparent rise in loss per event is mostly a change in what was recorded | BQ2 | OLAP 02 · 2.1–2.5 |
| F5 | Summer carries most loss; hail makes October–December a second peak | BQ4 | OLAP 04 · 4.1, 4.2, 4.5 |
| F6 | Each hazard creates a different claims workload | BQ4 | OLAP 04 · 4.3, 4.4 |
| F7 | The data does not support "lower income means lower insured loss" | BQ5 | OLAP 05 · 5.2, 5.3 |
| F8 | A fifth of the state analysis rests on inferred states; claims data is thin | Data quality | OLAP 06 |

The numbers in the text were taken from the run of 24 Nov 2026. The values printed under each chart are read live from the tables, so check they still match after any re-run.

In [ ]:
%run ./00_olap_setup

In [ ]:
NEEDED = ["bq1_rollup_hazard_state", "bq1_pareto_events", "bq1_annual_aggregate", "bq1_annual_aggregate_stats",
          "bq2_recording_threshold", "bq2_like_for_like", "bq3_state_per_resident", "bq3_dice_east_coast_warm",
          "bq4_pivot_events_month_hazard", "bq4_pivot_loss_month_hazard", "bq4_season_summary", "bq4_claims_by_hazard",
          "bq4_claims_season_line", "bq5_income_vs_loss", "bq5_correlation", "dq_cleaning_impact", "dq_state_source",
          "dq_measure_completeness", "dq_unplaced_events"]
missing = [t for t in NEEDED if not spark.catalog.tableExists(otname(t))]
assert not missing, f"Run OLAP notebooks 01–06 first (or 00_run_olap). Missing: {missing}"

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

EVIDENCE_DIR = "./evidence/olap"
try:
    os.makedirs(EVIDENCE_DIR, exist_ok=True)
except Exception:
    EVIDENCE_DIR = None

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "font.size": 9, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlecolor": INK, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GREY, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 8,
})

def finish(fig, name, source):
    fig.text(0.01, -0.02, f"Source: {source}", fontsize=7, color=MUTED, ha="left", va="top")
    fig.tight_layout()
    if EVIDENCE_DIR:
        try:
            fig.savefig(f"{EVIDENCE_DIR}/{name}.png", bbox_inches="tight")
        except Exception as e:
            print("Chart not saved:", e)
    plt.show()

def bn(x, _=None):
    return f"${x:,.0f}bn" if abs(x) >= 10 else f"${x:,.1f}bn"

---
## F1 · Flood and hail cause almost two-thirds of insured loss (BQ1)

In [ ]:
r = oread("bq1_rollup_hazard_state")
hz = r[r.level == "2 · family > hazard"].sort_values("loss_m")
fig, ax = plt.subplots(figsize=(8, 3.0))
ax.barh(hz.hazard_group, hz.loss_m / 1000, color=BLUE, height=0.6)
for y, (v, p, n) in enumerate(zip(hz.loss_m / 1000, hz.pct_of_total_loss, hz.events)):
    ax.text(v, y, f"  ${v:,.1f}bn · {p:.0f}% · {n} events", va="center", fontsize=8, color=INK_2)
ax.set_xlim(0, hz.loss_m.max() / 1000 * 1.55); ax.xaxis.set_major_formatter(mticker.FuncFormatter(bn))
ax.grid(axis="y", visible=False)
ax.set_title("Normalised insured loss by hazard, 2011–2026")
finish(fig, "f1_loss_by_hazard", "olap.bq1_rollup_hazard_state (ICA, 2022 AUD)")
print(hz[["hazard_group", "events", "loss_m", "pct_of_total_loss"]].sort_values("loss_m", ascending=False).to_string(index=False))

**Finding.** Floods ($21.9bn, 39%) and hail ($13.8bn, 24.5%) cause **63.5% of all loss from only 45 of 161 events**. Cyclones and storms are the most frequent hazards (84 events between them) but cause 28% of loss, because most of them are small. Queensland leads hail (58% of hail loss), cyclone (88%) and storm; New South Wales leads flood (44.5%).

**Recommendation.** Make **flood and hail in Queensland and New South Wales** the first priority for catastrophe reinsurance and accumulation monitoring. Frequent cyclone and storm events matter for the claims workload (F6) more than for reinsurance.

---
## F2 · A dozen events make half the loss, and a bad year is more than twice an average one (BQ1)

In [ ]:
p = oread("bq1_pareto_events").sort_values("loss_rank")
y = oread("bq1_annual_aggregate").sort_values("calendar_year")
s = oread("bq1_annual_aggregate_stats").iloc[0]
n50 = int(p[p.cumulative_pct >= 50].loss_rank.min()); n80 = int(p[p.cumulative_pct >= 80].loss_rank.min())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1, 1.3]})
a1.plot(p.loss_rank, p.cumulative_pct, color=BLUE, linewidth=2)
for n, t in [(n50, 50), (n80, 80)]:
    a1.plot([n], [t], "o", color=BLUE, markersize=6, markeredgecolor="#fcfcfb", markeredgewidth=1.5)
    a1.annotate(f"{n} events → {t}% of loss", (n, t), xytext=(8, -12), textcoords="offset points", fontsize=8, color=INK_2)
a1.set(xlabel="events ranked by loss (largest first)", ylim=(0, 102))
a1.yaxis.set_major_formatter(mticker.PercentFormatter())
a1.set_title(f"Cumulative share of loss ({len(p)} events with a loss)")

full = y[y.calendar_year < SCOPE_END_YEAR]
a2.bar(full.calendar_year, full.loss_m / 1000, color=BLUE, width=0.7)
part = y[y.calendar_year == SCOPE_END_YEAR]
a2.bar(part.calendar_year, part.loss_m / 1000, color=BLUE_LIGHT, width=0.7)
a2.axhline(s.mean_annual_loss_m / 1000, color=INK_2, linewidth=1, linestyle=(0, (4, 3)))
a2.text(2011.6, s.mean_annual_loss_m / 1000 + 0.2, f"mean ${s.mean_annual_loss_m / 1000:,.1f}bn", fontsize=8, color=INK_2)
a2.yaxis.set_major_formatter(mticker.FuncFormatter(bn)); a2.grid(axis="x", visible=False)
a2.set_title("Annual aggregate loss (2026 = Jan–Mar only, lighter)")
finish(fig, "f2_concentration_and_annual_loss", "olap.bq1_pareto_events, olap.bq1_annual_aggregate")
print(f"{n50} events = 50% of loss, {n80} events = 80%. Annual loss over {int(s.full_years)} full years: mean ${s.mean_annual_loss_m:,.0f}m, "
      f"median ${s.median_annual_loss_m:,.0f}m, std dev ${s.stddev_annual_loss_m:,.0f}m, worst {int(s.worst_year)} ${s.worst_year_loss_m:,.0f}m, "
      f"{int(s.years_above_twice_mean)} years above twice the mean.")

**Finding.** The largest **12 events (9% of events with a loss) cause half of all loss**, and 28 events (22%) cause 80%. The single largest, the February 2022 South-East Queensland and NSW floods, is 11% of 15 years of loss. Annual loss averaged **$3.7bn** (median $2.7bn, standard deviation $2.4bn). Two of 15 full years, 2011 and 2022, exceeded **$8bn**, more than twice the average: roughly one year in seven.

**Recommendation.** Loss is driven by rare large events, not the steady flow of small ones, so the main protection should be **per-event catastrophe excess-of-loss cover**, with an **aggregate cover** for a year like 2011 or 2022 when several large events land together. As a planning test, Kestrel should check that its programme survives a repeat of 2022 (one $6.4bn industry flood plus $1.9bn of other events) scaled to its market share.

---
## F3 · Queensland and NSW carry 77% of loss; Queensland is the outlier per resident (BQ3)

In [ ]:
st = oread("bq3_state_per_resident")
dice = oread("bq3_dice_east_coast_warm")
known = st[st.state_code != "UNK"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
t = st.sort_values("loss_m")
a1.barh(t.state_code, t.loss_m / 1000, color=[GREY if c == "UNK" else BLUE for c in t.state_code], height=0.6)
for yv, (v, pct) in enumerate(zip(t.loss_m / 1000, t.pct_of_all_loss)):
    a1.text(v, yv, f"  {pct:.0f}%", va="center", fontsize=8, color=INK_2)
a1.xaxis.set_major_formatter(mticker.FuncFormatter(bn)); a1.grid(axis="y", visible=False)
a1.set_title("Total loss by state (grey = state unknown)")
t = known.sort_values("loss_per_resident_aud")
a2.barh(t.state_code, t.loss_per_resident_aud, color=BLUE, height=0.6)
for yv, v in enumerate(t.loss_per_resident_aud):
    a2.text(v, yv, f"  ${v:,.0f}", va="center", fontsize=8, color=INK_2)
a2.set_xlim(0, known.loss_per_resident_aud.max() * 1.25); a2.grid(axis="y", visible=False)
a2.xaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
a2.set_title("Loss per resident, 2011–2026")
finish(fig, "f3_state_loss_per_resident", "olap.bq3_state_per_resident (ICA loss ÷ ABS population)")
print(st[["state_code", "events", "loss_m", "loss_per_resident_aud", "annual_loss_per_resident_aud", "pct_of_all_loss"]].to_string(index=False))
print(dice.to_string(index=False))

**Finding.** Queensland (49%) and NSW (28%) carry **77% of all loss**. Per resident, Queensland's loss is **$5,259, 2.7 times NSW ($1,932) and 5.6 times Victoria ($934)**, about $330 a resident every year. The Northern Territory has the most events per resident (15.9 per million) but they are small. The ACT has no recorded events. One sub-cube stands out: **Queensland and NSW weather events in spring and summer are 25% of events but 58% of loss**.

**Recommendation.** Review **Queensland rates across all weather perils first, then NSW flood**. Pricing and accumulation limits should be set by state *and* season, because the spring–summer east-coast cell carries most of the risk.

---
## F4 · The apparent rise in loss per event is mostly a change in what was recorded (BQ2)

In [ ]:
th = oread("bq2_recording_threshold").sort_values("calendar_year")
lf = oread("bq2_like_for_like").sort_values("period")
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3.1), gridspec_kw={"width_ratios": [1.2, 1.2, 1]})
a1.bar(th.calendar_year, th.events, color=BLUE, width=0.7); a1.grid(axis="x", visible=False)
for ax_ in (a1, a2):
    ax_.xaxis.set_major_locator(mticker.MultipleLocator(3)); ax_.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
a1.set_title("Events recorded per year")
a2.bar(th.calendar_year, th.events_under_100m, color=BLUE, width=0.7); a2.grid(axis="x", visible=False)
a2.set_title("…of which under $100m")
a3.bar(lf.period, lf.loss_per_year_m / 1000, color=BLUE, width=0.6)
for xv, (v, n) in enumerate(zip(lf.loss_per_year_m / 1000, lf.large_events_per_year)):
    a3.text(xv, v, f"${v:,.1f}bn\n{n} events/yr", ha="center", va="bottom", fontsize=8, color=INK_2)
a3.set_ylim(0, lf.loss_per_year_m.max() / 1000 * 1.3); a3.yaxis.set_major_formatter(mticker.FuncFormatter(bn))
a3.grid(axis="x", visible=False); a3.set_title("Like-for-like: $100m+ events, loss per year")
finish(fig, "f4_recording_threshold_and_trend", "olap.bq2_recording_threshold, olap.bq2_like_for_like")
print(th.to_string(index=False)); print(lf.to_string(index=False))

**Finding.** On the raw numbers, loss per event looks like it has **tripled**: **$312m per event over 2011–2019 against $991m over 2020–2025**. But the source changed what it records. Up to 2019 every year includes small events (4–19 events under $100m each year). From 2020 almost none are listed, and the count falls from 7–23 events a year to 2–6. Fewer small events in the denominator pushes the average up even if catastrophes are no worse.

Comparing only events of **$100m or more**, which would have been recorded in any year, there is **no clear trend**: 4.4, 3.0 and 4.2 such events a year, and $3.6bn, $3.3bn and $4.0bn of loss a year across 2011–15, 2016–20 and 2021–25. The latest period is about 10% above the first, which is within normal year-to-year variation (F2).

**Recommendation.** **Do not apply a premium trend loading based on the raw average loss per event**: it would overprice. Track the like-for-like $100m+ measure each year. Fifteen years of this data cannot separate a modest climate trend from noise, so any trend loading should come from external catastrophe models, not from this series alone.

---
## F5 · Summer carries most of the loss; hail makes October–December a second peak (BQ4)

In [ ]:
ev_m = oread("bq4_pivot_events_month_hazard").sort_values("month_number").fillna(0)
ls_m = oread("bq4_pivot_loss_month_hazard").sort_values("month_number").fillna(0)
seas = oread("bq4_season_summary")
cols = [h.lower() for h in HAZARD_ORDER]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, df, scale, title in [(a1, ev_m, 1, "Events by start month"), (a2, ls_m, 1000, "Normalised loss by start month")]:
    bottom = np.zeros(len(df))
    for h, c in zip(HAZARD_ORDER, cols):
        vals = df[c].astype(float).values / scale
        ax.bar(df.month_name.str[:3], vals, bottom=bottom, color=HAZARD_COLOURS[h], width=0.72,
               label=h, edgecolor="#fcfcfb", linewidth=1)
        bottom += vals
    ax.set_title(title); ax.grid(axis="x", visible=False); ax.tick_params(axis="x", labelsize=7.5)
a2.yaxis.set_major_formatter(mticker.FuncFormatter(bn))
handles, labels = a1.get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.06))
finish(fig, "f5_seasonality_month_hazard", "olap.bq4_pivot_events_month_hazard, olap.bq4_pivot_loss_month_hazard")
print(seas.to_string(index=False))
hail_q4 = ls_m[ls_m.month_number.isin([10, 11, 12])].hail.sum() / ls_m.hail.sum()
flood_jf = ls_m[ls_m.month_number.isin([1, 2])].flood.sum() / ls_m.flood.sum()
print(f"Hail loss in Oct–Dec: {100 * hail_q4:.0f}% · Flood loss in Jan–Feb: {100 * flood_jf:.0f}%")

**Finding.** **Summer (December–February) has 48% of events and 56% of loss**, and January is the busiest month (32 of 161 events). Spring is second by loss (26%) despite fewer events, because **69% of hail loss falls in October–December**. **69% of flood loss starts in January–February**, at the height of the wet season. Winter is quiet: 7.5% of events and 2.5% of loss.

**Recommendation.** Plan **surge claims capacity from October to March**, in two waves: hail and storm from October to December, then flood and cyclone from January to March. Leave and training should be scheduled in winter. Reserves and reinsurance liquidity should be at full strength by October.

---
## F6 · Each hazard creates a different claims workload (BQ4)

In [ ]:
ch = oread("bq4_claims_by_hazard")
ch = ch[ch.events_with_claims > 0].sort_values("loss_per_claim_aud")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 2.9))
a1.barh(ch.hazard_group, ch.loss_per_claim_aud, color=BLUE, height=0.6)
for yv, v in enumerate(ch.loss_per_claim_aud):
    a1.text(v, yv, f"  ${v:,.0f}", va="center", fontsize=8, color=INK_2)
a1.set_xlim(0, ch.loss_per_claim_aud.max() * 1.3); a1.grid(axis="y", visible=False)
a1.xaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}")); a1.set_title("Average loss per claim")
a2.barh(ch.hazard_group, ch.pct_motor_claims, color=BLUE, height=0.6)
for yv, v in enumerate(ch.pct_motor_claims):
    a2.text(v, yv, f"  {v:.0f}%", va="center", fontsize=8, color=INK_2)
a2.set_xlim(0, 70); a2.grid(axis="y", visible=False); a2.xaxis.set_major_formatter(mticker.PercentFormatter())
a2.set_title("Share of claims that are motor")
finish(fig, "f6_claims_profile_by_hazard", "olap.bq4_claims_by_hazard (events with a claim count only)")
print(ch.to_string(index=False))
print(oread("bq4_claims_season_line").to_string(index=False))

**Finding.** Bushfire claims are few but expensive (**about $70,000 per claim**, many total losses), cyclone about $32,000. Hail, storm and flood claims cost **$15,000–$20,000** each but come in volume: **42,000–61,000 claims per event** for flood and hail. **Over half of hail claims are motor claims**, and spring has almost four times as many motor claims as autumn. These figures use only the 46 events that record claim counts, so treat them as indicative (F8).

**Recommendation.** Staff by hazard, not just by month. For hail season, line up **motor assessors and repairer capacity** (October–January). For floods, line up **building assessors and builder panels** in Queensland and NSW (January–March). For bushfire and cyclone, set **higher initial case reserves per claim** and assign senior loss adjusters, because a few claims carry large values.

---
## F7 · The data does not support "lower income means lower insured loss" (BQ5)

In [ ]:
inc = oread("bq5_income_vs_loss")
cr = oread("bq5_correlation").sort_values("ord")
r_all = float(cr.iloc[0].r_income_vs_loss_per_resident)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.scatter(inc.avg_median_income_aud / 1000, inc.loss_per_resident_aud, s=60, color=BLUE, edgecolor="#fcfcfb", linewidth=1.5, zorder=3)
for _, row in inc.iterrows():
    ax.annotate(row.state_code, (row.avg_median_income_aud / 1000, row.loss_per_resident_aud), xytext=(6, 2),
                textcoords="offset points", fontsize=8, color=INK_2)
ax.set(xlabel="average median income ($k a year)", ylabel="loss per resident, 2011–2026")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.set_title(f"Income vs loss per resident (r = {r_all:.2f}, 8 states)")
finish(fig, "f7_income_vs_loss", "olap.bq5_income_vs_loss (ABS income 2019–2023, nearest-year filled)")
print(cr.drop(columns="ord").to_string(index=False))

**Finding.** The underinsurance idea predicts a **positive** link: poorer states insure less, so record less insured loss per person. The data shows the **opposite sign**: r = −0.37 for loss per resident and −0.50 for claims per 1,000 residents. Removing the ACT (highest income, no events) weakens it to −0.23. With eight points and the result swinging when one state is dropped, **neither direction is reliable**. The pattern is explained by **where hazards strike**: Queensland has lower-than-average income and the highest hazard exposure. Income figures are also only measured for 2019–2023, and filled from the nearest year elsewhere.

**Recommendation.** **Do not build an affordability or product strategy on state-level income from this data.** To test underinsurance properly, Kestrel should use its own policy data (policies, sums insured and claims by postcode) against ABS income at postcode level. That is a natural next phase for the warehouse: a `fact_policy` table at postcode grain using the same `dim_location`.

---
## F8 · A fifth of the state analysis rests on inferred states, and claims data is thin (data quality)

In [ ]:
cl = oread("dq_cleaning_impact")
src = oread("dq_state_source")
comp = oread("dq_measure_completeness")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 2.6), gridspec_kw={"width_ratios": [1, 1.3]})
a1.barh(cl.stage[::-1], cl.pct_loss_placed[::-1], color=[BLUE, BLUE_LIGHT], height=0.55)
for yv, v in enumerate(cl.pct_loss_placed[::-1]):
    a1.text(v, yv, f"  {v:.1f}%", va="center", fontsize=8, color=INK_2)
a1.set_xlim(0, 115); a1.xaxis.set_major_formatter(mticker.PercentFormatter()); a1.grid(axis="y", visible=False)
a1.set_title("Share of loss placed in a state")
left = 0
src_colours = {"reported": "#1c5cab", "inferred": "#6da7ec", "unknown": GREY}
for _, row in src.iterrows():
    a2.barh(["loss"], [row.pct_loss], left=left, color=src_colours[row.state_source], height=0.5, edgecolor="#fcfcfb", linewidth=2)
    a2.text(left + row.pct_loss / 2, 0, f"{row.state_source}\n{row.pct_loss:.0f}%", ha="center", va="center", fontsize=8,
            color="#ffffff" if row.state_source != "unknown" else INK)
    left += row.pct_loss
a2.set_xlim(0, 100); a2.axis("off"); a2.set_title("How each dollar of loss got its state")
finish(fig, "f8_data_quality", "olap.dq_cleaning_impact, olap.dq_state_source")
print(src.to_string(index=False)); print(comp.to_string(index=False))
print(oread("dq_unplaced_events").head(8).to_string(index=False))

**Finding.** Cleaning raised the share of loss that can be placed in a state from **8.7% to 93.9%**: without it, F3 and F7 could not be answered at all. But **20.7% of loss relies on a state inferred from the event name or towns**, and **6.1% ($3.4bn) still has no state**. Several unplaced events name a town that clearly belongs to one state (Tathra, NSW; Yarloop and Esperance, WA; Tasman Peninsula, TAS; Grampians and Geelong, VIC). Claim counts exist for only **29% of events** (7% of cyclones), and casualty data for 2%. Declared catastrophes are 46% of events but **97.8% of loss**.

**Recommendation.** (1) Add the missing town names to the keyword crosswalk (`silver.xw_state_keyword`) and re-run: this is cheap and would place about $0.5bn of loss in the right state from the town names alone. (2) Kestrel should **record state and claim counts against every catastrophe code** in its own claims system, so future analysis doesn't depend on inference. (3) For reporting and reinsurance, **declared catastrophes are a good proxy for total cat loss**: undeclared events add only 2%.

---
## Recommendations for Kestrel General Insurance

| # | Recommendation | Owner at Kestrel | Based on | Timing |
|---|---|---|---|---|
| R1 | Prioritise reinsurance and accumulation limits for **flood and hail in QLD and NSW** | Reinsurance and risk manager | F1, F3 | Next renewal |
| R2 | Buy **per-event catastrophe excess-of-loss** cover plus an **aggregate cover**; stress-test against a repeat of 2022 | Reinsurance and risk manager | F2 | Next renewal |
| R3 | Review **Queensland rates for all weather perils**, then **NSW flood**; set accumulation limits by state *and* season | Head of Underwriting | F3 | Next pricing cycle |
| R4 | **No trend loading** from the raw per-event average; track the like-for-like $100m+ measure yearly and take any trend from external cat models | Head of Underwriting | F4 | Ongoing |
| R5 | **Surge claims capacity October–March** in two waves: hail/storm (Oct–Dec), then flood/cyclone (Jan–Mar); leave and training in winter | Claims operations manager | F5 | Before October |
| R6 | Staff by hazard: **motor assessors** for hail, **builder panels** for flood in QLD/NSW, **higher initial reserves** for bushfire and cyclone claims | Claims operations manager | F6 | Before October |
| R7 | **Don't use state income for affordability decisions**; test underinsurance with Kestrel's own postcode-level policy data | Catastrophe pricing analyst | F7 | Next warehouse phase |
| R8 | Fix the keyword crosswalk; **record state and claim counts against every cat code** in Kestrel's own system | Data / claims systems owner | F8 | Immediately (crosswalk); next quarter (capture) |

### Limits on these conclusions
- **The data is industry-wide, not Kestrel's own.** Every dollar figure is market loss. Kestrel would scale it by its market share in each state and line.
- **Normalisation stops at 2022.** Losses from 2023 on are in their own year's dollars, so they are slightly overstated compared with earlier years. This does not change any finding.
- **Primary state only.** Multi-state events are placed in their first listed state, which understates NSW and VIC for events like the 2019/20 bushfires.
- **Small samples.** Eight states for F7; 46 events with claim counts for F6. Both are labelled indicative.